In [1]:
import krippendorff
from IPython.display import display, Markdown
from sklearn.metrics import cohen_kappa_score
import pandas as pd

In [2]:
# read quality control annotated_usages
df_annotator1 = pd.read_csv(
    "../../data/dev3/quality_control/euan_wei/6/usage_sample.tsv", sep="\t", encoding="utf-8", dtype=str)
df_annotator2 = pd.read_csv("../../data/dev3/quality_control/nash/6/usage_sample.tsv", sep="\t", encoding="utf-8", dtype=str)
df_annotator3 = pd.read_csv(
    "../../data/dev3/quality_control/johannes/6/usage_sample.tsv", sep="\t", encoding="utf-8", dtype=str).head(100)

 # remove one row that wasn't annotated
df_annotator3 = df_annotator3.dropna(subset=["sense_id"])
df_annotator3["sense_id"] = df_annotator3["sense_id"].astype(str)

# process annotated_usages
processed = []
for i, df in enumerate([df_annotator1, df_annotator2, df_annotator3], start=1):
    df = df[["identifier", "sense_id"]].copy()
    df["sense_id"] = df["sense_id"].apply(lambda a: a.split(",")[0].strip())  # only use primary annotation
    df["sense_id"] = df["sense_id"].apply(lambda a: "u" if a.startswith("-") else a)  # generalize unrecorded senses
    df = df.rename(columns={"sense_id": f"annotator{i}"})
    processed.append(df)
df_annotator1, df_annotator2, df_annotator3 = processed

# merge annotated_usages
df_annotations = df_annotator1.merge(df_annotator2, on="identifier")
df_annotations = df_annotations.merge(df_annotator3, on="identifier")

# read dictionary to extract sense_hierarchy information for each sense (for main-/sub-sense distinction)
df_dict = pd.read_csv("../../data/dev3/quality_control/euan_wei/6/dict.tsv", sep="\t", encoding="utf-8", dtype=str)
df_dict = df_dict[df_dict["source"]=="Lemur1000 ODENOAD Entries.xml"]
df_dict = df_dict[["sense_hierarchy", "sense_id"]].copy()

for i in [1, 2, 3]:
    # use sense hierarchy from dictionary to get main senses from annotated_usages
    df_annotations = df_annotations.merge(df_dict, left_on=f"annotator{i}", right_on="sense_id", how="left").drop(columns=["sense_id"]).rename(columns={"sense_hierarchy": f"main{i}"})
    df_annotations[f"main{i}"] = df_annotations.apply(lambda r: r[f"annotator{i}"] if pd.isna(r[f"main{i}"]) else r[f"main{i}"], axis=1).astype(str)
    df_annotations[f"main{i}"] = df_annotations[f"main{i}"].apply(lambda s: s.split(".")[0])

    # get type of annotated sense_id (LEMUR/Recorded/Unrecorded)
    df_annotations[f"type{i}"] = df_annotations[f"annotator{i}"].apply(lambda a: "unrecorded" if a == "u" else ("lemur" if a == "14" else "recorded"))

    # get boolean whether annotated sense_id is LEMUR sense or not
    df_annotations[f"is_lemur{i}"] = df_annotations[f"type{i}"].apply(lambda t: t == "lemur")

    # get boolean whether annotated sense_id is a recorded sense or not
    df_annotations[f"is_recorded{i}"] = df_annotations[f"type{i}"].apply(lambda t: t == "recorded")

display(df_annotations[["identifier", "annotator1", "annotator2", "annotator3", "main1", "main2", "main3", "type1", "type2", "type3", "is_lemur1", "is_lemur2", "is_lemur3", "is_recorded1", "is_recorded2", "is_recorded3"]].head(4))

,identifier,annotator1,annotator2,annotator3,main1,main2,main3,type1,type2,type3,is_lemur1,is_lemur2,is_lemur3,is_recorded1,is_recorded2,is_recorded3
0,NOW-1606NZ-9159209-3928529471,u,u,u,u,u,u,unrecorded,unrecorded,unrecorded,False,False,False,False,False,False
1,NOW-2402CA-106162196-31624267642,u,5,5,u,noun:3,noun:3,unrecorded,recorded,recorded,False,False,False,False,True,True
2,NOW-2207US1-89610314-25301314849,14,14,14,14,14,14,lemur,lemur,lemur,True,True,True,False,False,False
3,NOW-1701IE-16597751-5644925907,12,12,12,verb:2,verb:2,verb:2,recorded,recorded,recorded,False,False,False,True,True,True


In [3]:
def print_metrics(name1: str, name2: str):
    df_clean = df_annotations[(~df_annotations[name1].isin(["x", "0"])) & (~df_annotations[name2].isin(["x", "0"]))]

    # where both assigned a recorded sense calculate cohens kappa and agreement
    df_recorded = df_clean[(~df_clean[name1].isin(["u", "14"])) & (~df_clean[name2].isin(["u", "14"]))] # 14 is sense ID of LEMUR sense for the reel data

    display(Markdown(f"## Agreement Metrics {name1} & {name2}"))
    display(Markdown("Sub-Sense Agreement")) # based on exact annotation id
    print(f"Cohen's Kappa: {cohen_kappa_score(y1=df_recorded[name1], y2=df_recorded[name2])}")
    print(f"Agreement %: {len(df_recorded[df_recorded[name1] == df_recorded[name2]])/len(df_recorded)}")

    display(Markdown("Main-Sense Agreement"))  # based on main sense level assignment (noun:3, noun:3.1, noun:3.3 are all considered the same)
    print(f"Cohen's Kappa: {cohen_kappa_score(y1=df_recorded[f'main{name1[-1:]}'], y2=df_recorded[f'main{name1[-1:]}'])}")
    print(f"Agreement %: {len(df_recorded[df_recorded[f'main{name1[-1:]}'] == df_recorded[f'main{name1[-1:]}']])/len(df_recorded)}")

    # over all rows agreement on whether a usage matches the LEMUR sense or not
    display(Markdown("LEMUR Y/N Agreement"))
    print(f"Cohen's Kappa: {cohen_kappa_score(y1=df_clean[f'is_lemur{name1[-1:]}'], y2=df_clean[f'is_lemur{name2[-1:]}'])}")
    print(f"Agreement %: {len(df_clean[df_clean[f'is_lemur{name1[-1:]}'] == df_clean[f'is_lemur{name2[-1:]}']])/len(df_clean)}")

    # over all rows agreement on whether a usage is assigned a recorded sense or not
    display(Markdown("Recorded Y/N Agreement"))
    print(f"Cohen's Kappa: {cohen_kappa_score(y1=df_clean[f'is_recorded{name1[-1:]}'], y2=df_clean[f'is_recorded{name2[-1:]}'])}")
    print(f"Agreement %: {len(df_clean[df_clean[f'is_recorded{name1[-1:]}'] == df_clean[f'is_recorded{name2[-1:]}']])/len(df_clean)}")

# calculate and print cohens kappa and agreement
print_metrics("annotator1", "annotator2")
print_metrics("annotator1", "annotator3")
print_metrics("annotator2", "annotator3")

# calculate krippendorff alpha
display(Markdown("## Krippendorff"))
df_1_2 = df_annotations[(~df_annotations["annotator1"].isin(["x", "0"])) & (~df_annotations["annotator2"].isin(["x", "0"]))]
df_1_2_only_recorded = df_1_2[(~df_1_2["annotator1"].isin(["u", "14"])) & (~df_1_2["annotator2"].isin(["u", "14"]))]
df_1_3 = df_annotations[(~df_annotations["annotator1"].isin(["x", "0"])) & (~df_annotations["annotator3"].isin(["x", "0"]))]
df_1_3_only_recorded = df_1_3[(~df_1_3["annotator1"].isin(["u", "14"])) & (~df_1_3["annotator3"].isin(["u", "14"]))]
df_2_3 = df_annotations[(~df_annotations["annotator2"].isin(["x", "0"])) & (~df_annotations["annotator3"].isin(["x", "0"]))]
df_2_3_only_recorded = df_2_3[(~df_2_3["annotator2"].isin(["u", "14"])) & (~df_2_3["annotator3"].isin(["u", "14"]))]
df_1_2_3 = df_annotations[(~df_annotations["annotator1"].isin(["x", "0"])) & (~df_annotations["annotator2"].isin(["x", "0"])) & (~df_annotations["annotator3"].isin(["x", "0"]))]
df_1_2_3_only_recorded = df_1_2_3[(~df_1_2_3["annotator1"].isin(["u", "14"])) & (~df_1_2_3["annotator2"].isin(["u", "14"])) & (~df_1_2_3["annotator3"].isin(["u", "14"]))]

sub_matrix = [
    df_1_2_3_only_recorded["annotator1"].tolist(),
    df_1_2_3_only_recorded["annotator2"].tolist(),
    df_1_2_3_only_recorded["annotator3"].tolist()
]
main_matrix = [
    df_1_2_3_only_recorded["main1"].tolist(),
    df_1_2_3_only_recorded["main2"].tolist(),
    df_1_2_3_only_recorded["main3"].tolist()
]
recorded_matrix = [
    df_1_2["is_recorded1"].astype(int).tolist(),
    df_1_2["is_recorded2"].astype(int).tolist(),
    df_1_2["is_recorded3"].astype(int).tolist()
]

lemur_matrix = [
    df_1_2["is_lemur1"].astype(int).tolist(),
    df_1_2["is_lemur2"].astype(int).tolist(),
    df_1_2["is_lemur3"].astype(int).tolist()
]
alpha_sub = krippendorff.alpha(reliability_data=sub_matrix, level_of_measurement='nominal')
alpha_main = krippendorff.alpha(reliability_data=main_matrix, level_of_measurement='nominal')
alpha_lemur = krippendorff.alpha(reliability_data=lemur_matrix, level_of_measurement='nominal')
alpha_recorded = krippendorff.alpha(reliability_data=recorded_matrix, level_of_measurement='nominal')
print(f"Krippendorff Alpha Sub-Sense: {alpha_sub}")
print(f"Krippendorff Alpha Main-Sense: {alpha_main}")
print(f"Krippendorff Alpha LEMUR Y/N: {alpha_lemur}")
print(f"Krippendorff Alpha Recorded Y/N: {alpha_recorded}")

## Agreement Metrics annotator1 & annotator2

Sub-Sense Agreement

Cohen's Kappa: 0.6153846153846154
Agreement %: 0.7


Main-Sense Agreement

Cohen's Kappa: 1.0
Agreement %: 1.0


...

Cohen's Kappa: 0.6654804270462633
Agreement %: 0.9148936170212766


Recorded Y/N Agreement

Cohen's Kappa: 0.6521953626048347
Agreement %: 0.8404255319148937


## Krippendorff

Krippendorff Alpha Sub-Sense: 0.6931447225244831
Krippendorff Alpha Main-Sense: 0.8879618593563766
Krippendorff Alpha LEMUR Y/N: 0.7201872201872201
Krippendorff Alpha Recorded Y/N: 0.21875
